In [1]:
import random
import time
import math
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.transforms import functional as FT
from PIL import Image, ImageChops
import cv2
from sklearn.metrics import cohen_kappa_score
import timm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True  # enable cudnn auto‑tuner for speed




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
threshold = [0.75, 1.5, 2.5, 3.5]


def regress2class(out):
    prediction = torch.zeros(out.size(0))
    for i in range(4):
        prediction += (out.data >= threshold[i]).squeeze().cpu()
    return prediction


def ordinal2class_prob(out):
    pred_prob = torch.zeros(out.size(0), 5).to(device)
    pred_prob[:, 0] = (1 - out[:, 0]).squeeze()
    pred_prob[:, 1] = (out[:, 0] * (1 - out[:, 1])).squeeze()
    pred_prob[:, 2] = (out[:, 1] * (1 - out[:, 2])).squeeze()
    pred_prob[:, 3] = (out[:, 2] * (1 - out[:, 3])).squeeze()
    pred_prob[:, 4] = out[:, 3].squeeze()
    return F.softmax(pred_prob, dim=1)


def regress2class_prob(out):
    out = out.squeeze()
    batch = out.size(0)
    floor = torch.floor(out).long()
    ceil = torch.ceil(out).long()
    prob = torch.zeros((batch, 5), device=out.device)

    mask_last = out >= 4.0
    not_last = ~mask_last

    f = floor[not_last]
    c = ceil[not_last]
    o = out[not_last]
    prob[not_last, f] = 1 - (o - f.float())
    prob[not_last, c] = 1 - (c.float() - o)

    prob[mask_last, 4] = 1.0

    return F.softmax(prob, dim=1)


def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p)


class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6, flatten=False):
        super(GeM, self).__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps
        self.flatten = flatten

    def forward(self, x):
        x = gem(x, p=self.p, eps=self.eps)
        if self.flatten:
            x = x.flatten(1)
        return x

    def __repr__(self):
        return (
            self.__class__.__name__
            + "("
            + "p="
            + "{:.4f}".format(self.p.data.tolist()[0])
            + ", "
            + "eps="
            + str(self.eps)
            + ")"
        )


class Regressor(nn.Module):
    def __init__(self):
        super(Regressor, self).__init__()
        self.backbone = timm.create_model("tf_efficientnet_b5_ns", pretrained=True)
        self.backbone.global_pool = GeM(flatten=True)
        self.regressor = nn.Linear(1000, 1)

    def forward(self, x):
        x = self.backbone(x)
        out = self.regressor(x)
        out = torch.sigmoid(out) * 4.0  # corrected scaling
        return out


class ThreeStage_Model(nn.Module):
    def __init__(self):
        super(ThreeStage_Model, self).__init__()
        self.backbone = timm.create_model("tf_efficientnet_b4_ns", pretrained=True)
        self.backbone.global_pool = GeM(flatten=True)

        self.classifier = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 5),
        )

        self.regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 1),
        )

        self.ordinal = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 4),
        )

        self.final_regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x, final=False):
        x = self.backbone(x)
        c_out = self.classifier(x)
        r_out = self.regressor(x)
        o_out = self.ordinal(x)
        if final:
            out = torch.cat((c_out, r_out, o_out), 1)
            out = self.final_regressor(out)
            out = torch.sigmoid(out) * 4.0  # corrected scaling
            return out
        else:
            r_out = torch.sigmoid(r_out) * 4.0  # corrected scaling
            o_out = torch.sigmoid(o_out)
            return c_out, r_out, o_out


class trim(object):
    def __call__(self, image):
        bg = Image.new(image.mode, image.size, image.getpixel((0, 0)))
        diff = ImageChops.difference(image, bg)
        diff = ImageChops.add(diff, diff, 2.0, -10)
        bbox = diff.getbbox()
        if bbox:
            return image.crop(bbox)
        return image


class cropTo4_3(object):
    def __call__(self, image):
        w, h = image.size
        if (w / h) >= (4 / 3):
            new_h = h
            new_w = int(h * 4 / 3)
        else:
            new_h = int(w * 3 / 4)
            new_w = w
        left = (w - new_w) / 2
        top = (h - new_h) / 2
        right = left + new_w
        bottom = top + new_h
        return image.crop((left, top, right, bottom))




In [3]:
input_size = 380
train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(
            (input_size * 3 // 4, input_size), scale=(0.8, 1.0)
        ),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.384, 0.258, 0.174], std=[0.124, 0.089, 0.094]),
    ]
)

test_transform = transforms.Compose(
    [
        trim(),
        cropTo4_3(),
        transforms.Resize((input_size * 3 // 4, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.384, 0.258, 0.174], std=[0.124, 0.089, 0.094]),
    ]
)


class AptosDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform):
        df = pd.read_csv(csv_path)
        self.samples = [
            (os.path.join(img_dir, f"{row['id_code']}.png"), int(row["diagnosis"]))
            for _, row in df.iterrows()
        ]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        return image, label


net = ThreeStage_Model()
weight_path = "../input/weights/B4_3stage_4epoch_finetune.pkl"
weights_loaded = False
try:
    net.load_state_dict(torch.load(weight_path, map_location=device))
    print("Loaded fine‑tuned weights.")
    weights_loaded = True
except Exception as e:
    print(
        f"Fine‑tuned weights not found or failed to load ({e}); will use pretrained model."
    )

net = net.to(device)
net.eval()

if not weights_loaded:
    print("Proceeding with pretrained model only (no additional fine‑tuning).")




/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Fine‑tuned weights not found or failed to load ([Errno 2] No such file or directory: '../input/weights/B4_3stage_4epoch_finetune.pkl'); will use pretrained model.
Proceeding with pretrained model only (no additional fine‑tuning).


In [4]:
def worker_init_fn(worker_id):
    worker_seed = seed + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


if not weights_loaded:
    print("Starting quick fine‑tuning on the training set (5 epochs).")
    train_csv = "../input/aptos2019-blindness-detection/train.csv"
    train_img_dir = "../input/aptos2019-blindness-detection/train_images"
    train_dataset = AptosDataset(train_csv, train_img_dir, train_transform)
    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,
        num_workers=os.cpu_count(),
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )

    optimizer = torch.optim.Adam(net.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    net.train()
    for epoch in range(5):  # increased epochs
        epoch_loss = 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            c_out, _, _ = net(imgs)  # use classifier output
            loss = criterion(c_out, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(
            f"Epoch {epoch+1}/5 finished, avg loss: {epoch_loss/len(train_loader):.4f}"
        )

    net.eval()
    print("Fine‑tuning completed.")




Starting quick fine‑tuning on the training set (5 epochs).


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch 1/5 finished, avg loss: 0.7411
Epoch 2/5 finished, avg loss: 0.4964
Epoch 3/5 finished, avg loss: 0.3737
Epoch 4/5 finished, avg loss: 0.2885
Epoch 5/5 finished, avg loss: 0.2508
Fine‑tuning completed.


In [5]:
class TestDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform):
        self.ids = pd.read_csv(csv_path)["id_code"].tolist()
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_code = self.ids[idx]
        img_path = os.path.join(self.img_dir, f"{id_code}.png")
        img = Image.open(img_path).convert("RGB")
        base = self.transform(img)  # (C, H, W)
        hflip = torch.flip(base, dims=[2])  # horizontal flip
        vflip = torch.flip(base, dims=[1])  # vertical flip
        return base, hflip, vflip, id_code


test_csv = "../input/aptos2019-blindness-detection/test.csv"
test_img_dir = "../input/aptos2019-blindness-detection/test_images"
test_dataset = TestDataset(test_csv, test_img_dir, test_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=os.cpu_count(),
    pin_memory=True,
    worker_init_fn=worker_init_fn,
)

submission = []
class_indices = torch.arange(5, dtype=torch.float32, device=device)

net.eval()
with torch.no_grad():
    for batch_idx, (orig, hflip, vflip, id_codes) in enumerate(test_loader):
        batch = torch.stack([orig, hflip, vflip], dim=1).to(device)  # (B,3,C,H,W)
        B, N, C, H, W = batch.shape
        batch_flat = batch.view(B * N, C, H, W)

        c_out, r_out, _ = net(batch_flat)  # (B*3,5) and (B*3,1)
        c_out = c_out.view(B, N, -1)  # (B,3,5)
        r_out = r_out.view(B, N, -1)  # (B,3,1)

        c_out_avg = c_out.mean(dim=1)  # (B,5)
        probs_clf = torch.softmax(c_out_avg, dim=1)

        r_out_avg = r_out.mean(dim=1).squeeze()  # (B,)
        probs_reg = regress2class_prob(r_out_avg)  # (B,5)

        # New inference strategy: rely solely on classifier probabilities.
        combined_probs = probs_clf  # previously 0.7*clf + 0.3*reg

        expected = torch.sum(combined_probs * class_indices, dim=1)
        pred_class = torch.clamp(torch.round(expected), 0, 4).int().cpu().numpy()

        for id_code, pred in zip(id_codes, pred_class):
            submission.append([id_code, int(pred)])

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(test_loader):
            print(
                f"Processed {(batch_idx + 1) * test_loader.batch_size} / {len(test_dataset)} images"
            )

submission = np.array(submission)

df = pd.DataFrame(submission, columns=["id_code", "diagnosis"])
if df.empty:
    print("Warning: submission is empty. Writing a fallback submission with class 0.")
    fallback = pd.read_csv(
        "../input/aptos2019-blindness-detection/sample_submission.csv"
    )
    fallback["diagnosis"] = 0
    df = fallback

df.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Processed 320 / 367 images
Processed 384 / 367 images
Submission saved to submission.csv
